In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["NEO4J_URI"]=os.getenv("NEO4J_URI")
os.environ["NEO4J_USERNAME"]=os.getenv("NEO4J_USERNAME")
os.environ["NEO4J_PASSWORD"]=os.getenv("NEO4J_PASSWORD")

In [ ]:
from langchain_neo4j import Neo4jGraph
graph=Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
)

This code snippet initializes a Neo4jGraph object from the langchain_community.graphs module within the LangChain library. This object allows you to interact with a Neo4j graph database. Let's break down each part:

1. from langchain_community.graphs import Neo4jGraph

This line imports the Neo4jGraph class from the langchain_community.graphs module.
Neo4jGraph is a LangChain integration that provides a convenient way to interact with a Neo4j graph database.
2. graph = Neo4jGraph(...)

This line creates an instance of the Neo4jGraph class and assigns it to the variable graph.

The constructor of the Neo4jGraph class takes the following arguments:

url=NEO4J_URI:
NEO4J_URI is a variable that should contain the URL of your Neo4j database.
This URL specifies the location of your Neo4j server.
It typically looks like bolt://<host>:<port>.
bolt is the protocol used to connect to Neo4j.
username=NEO4J_USERNAME:
NEO4J_USERNAME is a variable that should contain the username for your Neo4j database.
password=NEO4J_PASSWORD:
NEO4J_PASSWORD is a variable that should contain the password for your Neo4j database.

In [ ]:
graph

This Cypher query is designed to load movie data from a CSV file hosted on GitHub and create a graph representation of that data in a Neo4j database. Let's break it down step by step:

In [ ]:
### Load the dataset of movie

movie_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') |
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') |
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') |
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))
"""

In summary, this Cypher query does the following:

Loads movie data from a CSV file.
Creates Movie nodes with properties like id, released, title, and imdbRating.
Creates Person nodes for directors and actors, and establishes DIRECTED and ACTED_IN relationships.
Creates Genre nodes and establishes IN_GENRE relationships between movies and genres.

1. LOAD CSV WITH HEADERS FROM 'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

LOAD CSV WITH HEADERS FROM ...: This command instructs Neo4j to load data from a CSV file.
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv': This is the URL of the CSV file containing the movie data. It's being loaded directly from a raw GitHub URL.
as row: This assigns each row of the CSV file to the variable row, allowing you to access the data in each column using row.columnName.
WITH HEADERS: Specifies that the first row of the CSV file contains column headers.
2. MERGE(m:Movie{id:row.movieId})

MERGE(m:Movie{id:row.movieId}): This command either creates a new Movie node or matches an existing one.
MERGE: Ensures that if a movie with the given movieId already exists, it won't be created again.
(m:Movie{id:row.movieId}): Creates a node with the label Movie and a property id set to the value of the movieId column from the CSV. The m is a variable representing the movie node.
3. SET m.released = date(row.released), m.title = row.title, m.imdbRating = toFloat(row.imdbRating)

SET: This command sets properties on the Movie node (m).
m.released = date(row.released): Sets the released property of the movie to a date value parsed from the released column.
m.title = row.title: Sets the title property of the movie to the value of the title column.
m.imdbRating = toFloat(row.imdbRating): Sets the imdbRating property of the movie to a floating-point number parsed from the imdbRating column.
4. FOREACH (director in split(row.director, '|') | ...)

FOREACH: This command iterates over a list.
split(row.director, '|'): This splits the director column (which likely contains multiple directors separated by '|') into a list of director names.
MERGE (p:Person {name:trim(director)}): For each director name:
MERGE (p:Person {name:trim(director)}): Creates a new Person node (or matches an existing one) with the name property set to the director's name (trimmed to remove leading/trailing whitespace).
MERGE (p)-[:DIRECTED]->(m)): Creates a DIRECTED relationship from the Person node (p) to the Movie node (m).
5. FOREACH (actor in split(row.actors, '|') | ...)

This part is similar to the director part, but it creates Person nodes for actors and ACTED_IN relationships between actors and movies.
6. FOREACH (genre in split(row.genres, '|') | ...)

This part is similar to the director and actor parts, but it creates Genre nodes and IN_GENRE relationships between movies and genres.

In [ ]:
graph.query(movie_query)

[]

In [ ]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Person {name: STRING}
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Person)-[:FRIENDS]->(:Person)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)
